<a href="https://colab.research.google.com/github/P4mell4/Telecom-X---2/blob/main/Telecom_02_pamella.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#🧩Challenge 03 Telecom X part 2

###Extração do Arquivo Tratado

In [ ]:
url = "https://raw.githubusercontent.com/P4mell4/Telecom-X---2/refs/heads/main/telecomx_tratado.csv"
import pandas as pd
df = pd.read_csv(url)

In [ ]:
#conferindo
df.head()

###Remoção de Colunas Irrelevantes

In [ ]:
df = df.drop(columns=["customerID"], errors="ignore")

In [ ]:
#conderindo
df.columns

###Encoding

In [ ]:
df_encoded = pd.get_dummies(df, drop_first=True)

In [ ]:
df_encoded.head()

In [ ]:
#conferinco
df_encoded.info()

###Verificação da Proporção de Evasão

**Contagem das classes**

In [ ]:
df_encoded["churn"].value_counts()

**Ver proporção em %**

In [ ]:
df_encoded["churn"].value_counts(normalize=True)

ou seja:


*   73.4% permaneceram
*   26.6% cancelaram



###Balanceamento de Classes

**Undersampling**

In [ ]:
from sklearn.utils import resample

classe_majoritaria = df_encoded[df_encoded["churn"] == 0]
classe_minoritaria = df_encoded[df_encoded["churn"] == 1]

classe_majoritaria_downsample = resample(
    classe_majoritaria,
    replace=False,
    n_samples=len(classe_minoritaria),
    random_state=42
)

df_balanceado = pd.concat([classe_majoritaria_downsample, classe_minoritaria])

**Oversampling**

In [ ]:
classe_minoritaria_upsample = resample(
    classe_minoritaria,
    replace=True,
    n_samples=len(classe_majoritaria),
    random_state=42
)

df_balanceado = pd.concat([classe_majoritaria, classe_minoritaria_upsample])

**SMOTE (mais avançado e pesquisado por fora)**

In [ ]:
#instalando a bibliotc
!pip install imbalanced-learn

In [ ]:
df_encoded = df_encoded.dropna(subset=["churn"])

In [ ]:
X = df_encoded.drop("churn", axis=1)
y = df_encoded["churn"]

In [ ]:
y.isna().sum()

In [ ]:
from imblearn.over_sampling import SMOTE

smote = SMOTE(random_state=42)

X_res, y_res = smote.fit_resample(X, y)

In [ ]:
y_res.value_counts()

###Normalização ou Padronização

In [ ]:
X = df_encoded.drop("churn", axis=1)
y = df_encoded["churn"]

In [ ]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()

X_scaled = scaler.fit_transform(X)

In [ ]:
#conferindo
X_scaled[:5]

###Separação de Dados (Treino e Teste)

In [ ]:
from sklearn.model_selection import train_test_split

X = df_encoded.drop("churn", axis=1)
y = df_encoded["churn"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

###Criação de Modelos

**Modelo 1 — Regressão Logística**

In [ ]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

In [ ]:
from sklearn.linear_model import LogisticRegression

log_model = LogisticRegression(max_iter=1000)

log_model.fit(X_train_scaled, y_train)

**Modelo 2 — Random Forest**

In [ ]:
from sklearn.ensemble import RandomForestClassifier

rf_model = RandomForestClassifier(random_state=42)

rf_model.fit(X_train, y_train)

###Avaliação dos Modelos

In [ ]:
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    classification_report
)

**Avaliação — Regressão Logística**

In [ ]:
y_pred_log = log_model.predict(X_test_scaled)

print("Acurácia:", accuracy_score(y_test, y_pred_log))
print("Precisão:", precision_score(y_test, y_pred_log))
print("Recall:", recall_score(y_test, y_pred_log))
print("F1-score:", f1_score(y_test, y_pred_log))

print("\nRelatório de Classificação:\n")
print(classification_report(y_test, y_pred_log))

**Matriz de Confusão**

In [ ]:
sns.heatmap(confusion_matrix(y_test, y_pred_log),
            annot=True,
            fmt="d",
            cmap="Blues")

plt.title("Matriz de Confusão - Regressão Logística")
plt.xlabel("Previsto")
plt.ylabel("Real")
plt.show()

**Avaliação — Random Forest**

In [ ]:
y_pred_rf = rf_model.predict(X_test)

print("Acurácia:", accuracy_score(y_test, y_pred_rf))
print("Precisão:", precision_score(y_test, y_pred_rf))
print("Recall:", recall_score(y_test, y_pred_rf))
print("F1-score:", f1_score(y_test, y_pred_rf))

print("\nRelatório de Classificação:\n")
print(classification_report(y_test, y_pred_rf))

**Matriz de Confusão**

In [ ]:
sns.heatmap(confusion_matrix(y_test, y_pred_rf),
            annot=True,
            fmt="d",
            cmap="Greens")

plt.title("Matriz de Confusão - Random Forest")
plt.xlabel("Previsto")
plt.ylabel("Real")
plt.show()

###Análise de Importância das Variáveis

**importância das Variáveis — Random Forest**

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

importances = rf_model.feature_importances_

feature_importance = pd.DataFrame({
    "Variável": X.columns,
    "Importância": importances
})

feature_importance = feature_importance.sort_values(by="Importância", ascending=False)

In [ ]:
feature_importance.head(10)

In [ ]:
plt.figure(figsize=(8,6))

sns.barplot(
    data=feature_importance.head(10),
    x="Importância",
    y="Variável"
)

plt.title("Top 10 Variáveis Mais Importantes para Prever Churn")
plt.show()

**Coeficientes — Regressão Logística**

In [ ]:
coeficientes = pd.DataFrame({
    "Variável": X.columns,
    "Coeficiente": log_model.coef_[0]
})

coeficientes = coeficientes.sort_values(by="Coeficiente", ascending=False)

coeficientes.head(10)

#📉Relatório de Análise — Previsão de Evasão de Clientes (Churn)

**Introdução**

Este projeto teve como objetivo desenvolver modelos de Machine Learning capazes de prever a evasão de clientes (churn) da Telecom X. A identificação antecipada de clientes com maior probabilidade de cancelamento permite que a empresa desenvolva estratégias de retenção mais eficazes.

Para isso, foi realizado um processo completo de preparação dos dados, análise exploratória, modelagem preditiva e interpretação dos resultados.

**Preparação dos Dados**

Durante o processo de preparação dos dados foram realizadas as seguintes etapas:

*   Remoção de colunas irrelevantes (como identificadores de cliente).

*   Codificação de variáveis categóricas utilizando One-Hot Encoding.

*   Verificação da proporção de evasão no dataset.

*   Aplicação de técnicas de balanceamento de classes utilizando SMOTE.

*   Padronização das variáveis numéricas para modelos sensíveis à escala.

Essas etapas foram fundamentais para garantir a qualidade dos dados e melhorar o desempenho dos modelos preditivos.

**Modelagem Preditiva**

Foram utilizados dois modelos de classificação para prever a evasão de clientes:

**Regressão Logística**

A regressão logística foi escolhida por ser um modelo amplamente utilizado em problemas de classificação binária. Como esse modelo é sensível à escala dos dados, foi aplicada padronização utilizando StandardScaler.

Random Forest

O Random Forest foi utilizado por sua capacidade de capturar relações não lineares entre as variáveis e por ser menos sensível à escala dos dados. Esse modelo também possui boa capacidade de generalização e fornece medidas de importância das variáveis.

**Avaliação dos Modelos**

s modelos foram avaliados utilizando as seguintes métricas:

*   Acurácia

*   Precisão

*   Recall

*   F1-score

*   Matriz de Confusão

De forma geral, o modelo Random Forest apresentou melhor desempenho, conseguindo capturar padrões mais complexos nos dados e apresentando maior equilíbrio entre precisão e recall.

Não foram observados sinais significativos de underfitting. O Random Forest pode apresentar tendência a overfitting em alguns casos, porém isso pode ser controlado ajustando parâmetros como profundidade das árvores e número de estimadores.

**Variáveis Mais Importantes para Prever Churn**

A análise de importância das variáveis revelou alguns fatores com forte influência na evasão de clientes.

As variáveis mais relevantes incluem:

**Tempo de Contrato (tenure)**

Clientes com menor tempo de contrato apresentam maior probabilidade de evasão. Isso indica que os primeiros meses de relacionamento com o cliente são críticos para retenção.

**Tipo de Contrato**

Clientes com contratos mensais (month-to-month) tendem a cancelar mais frequentemente do que clientes com contratos anuais ou bienais.

**Valor Mensal do Serviço (MonthlyCharges)**

Clientes com cobranças mensais mais altas apresentam maior tendência de cancelamento, possivelmente devido à percepção de custo elevado em relação ao valor entregue.

**Valor Total Gasto (TotalCharges)**

Clientes com maior investimento total na empresa tendem a permanecer por mais tempo, indicando maior fidelização.

**Serviços Contratados**

Alguns serviços adicionais também apresentam influência na evasão, sugerindo que a composição do pacote contratado pode afetar a satisfação do cliente.

**Principais Fatores que Influenciam a Evasão**

Com base na análise dos dados e nos modelos preditivos, os principais fatores associados ao churn são:

*   Baixo tempo de permanência na empresa

*   Contratos mensais

*   Altos custos mensais

*   Baixo valor total gasto ao longo do tempo

*   Ausência de serviços adicionais que aumentem o valor percebido

Esses fatores indicam que clientes recém-adquiridos e com contratos mais flexíveis apresentam maior risco de evasão.

**Estratégias de Retenção Recomendadas**

Com base nos resultados da análise, algumas estratégias podem ser sugeridas para reduzir a evasão de clientes:

*   Programas de fidelização

*   Incentivar contratos de longo prazo oferecendo descontos ou benefícios para planos anuais ou bienais.

*   Melhor acompanhamento de novos clientes

*   Implementar estratégias de acompanhamento nos primeiros meses de contrato, período em que o risco de evasão é maior.

*   Revisão da estrutura de preços

*   Avaliar a percepção de valor dos clientes com altos custos mensais, oferecendo pacotes mais competitivos ou benefícios adicionais.

*   Ofertas personalizadas

*   Utilizar modelos preditivos para identificar clientes com maior risco de churn e oferecer promoções ou melhorias no serviço antes do cancelamento.

**Conclusão**

O uso de modelos de Machine Learning demonstrou ser uma abordagem eficaz para prever a evasão de clientes na Telecom X.

Os resultados obtidos permitem identificar padrões importantes no comportamento dos clientes e fornecer insights estratégicos para a empresa. A implementação de estratégias baseadas nesses insights pode contribuir significativamente para a redução da taxa de churn e para o aumento da fidelização dos clientes.